# 🧪 W4-D1 概念实验：RAG 四步管道到底在干什么？

> 配套阅读：`ima/第4周-Day1-RAG基本流程与架构.md`（完整原理、组件技术栈、常见误区在那边）
>
> 这个 notebook 只做一件事：**把"切割 → 向量化 → 检索 → 增强生成"四步管道跑成可执行的代码**，
> 并用实验回答两个问题：
> 1. 为什么闭卷 LLM 会答错店里最新的价格？（知识截止与幻觉的根源）
> 2. 切割（chunking）做坏了，检索会发生什么？
>
> 实验环境：纯 Python + numpy 模拟，不调用任何真实 LLM / Embedding API。

## 实验 1：闭卷 LLM vs 开卷 RAG —— 知识截止是怎么产生的

模拟一个"知识截止于 2024 年"的 LLM：它内部只记得旧菜单。
店里 2026 年的真实菜单（含全新产品）它从没见过。
闭卷作答时，模型只能拿旧知识硬套，或者干脆编一个——这就是幻觉的雏形。

In [ ]:
# 模拟一个"训练数据截止于 2024"的 LLM：内部菜单是旧的
llm_internal_knowledge = {
    "杨枝甘露": "16 元（2024 旧菜单价）",
    "芒果双皮奶": "12 元（2024 旧菜单价）",
}
# 店里 2026 年的真实菜单 —— LLM 训练时根本没见过
store_menu_2026 = {
    "杨枝甘露": "22 元",
    "芒果双皮奶": "15 元",
    "芋泥波波冰": "18 元",   # 全新产品，旧模型连名字都没见过
}

question = "现在店里的芋泥波波冰卖多少钱？"

def closed_book_llm(q):
    # 闭卷：只靠训练时的旧知识硬答
    for name in llm_internal_knowledge:
        if name[0] in q:
            return f"大概是 {llm_internal_knowledge[name]} 吧"
    return "没听过这个产品……估计 12 元左右？（按同类产品瞎估）"

print(f"顾客提问：{question}")
print(f"闭卷 LLM 回答：{closed_book_llm(question)}")
print(f"真实答案　　：{store_menu_2026['芋泥波波冰']}")
print()
print("→ 模型没见过 2026 菜单，只能拿旧价硬套或编造。")
print("→ RAG 的思路：别让模型背知识，提问时现场查资料再回答。")

## 实验 2：RAG 四步管道最小可运行版

用字符 bigram + TF-IDF 模拟"向量化"（真实系统用 Embedding 模型，思想一致：把文本变成可比较的向量），
把四步管道完整跑一遍：**切割 → 向量化 → 余弦检索 Top-K → 带上下文抽取答案**。

In [ ]:
import re
import numpy as np

# ---- Step 0: 原始文档（模拟知识库）----
docs = [
    "本店招牌杨枝甘露使用泰国椰浆与新鲜芒果，冰镇出品，夏季人气第一，售价22元。",
    "杨枝甘露2026年新配方加入西柚果粒，微酸解腻。",
    "双皮奶采用顺德水牛奶隔水蒸制，奶皮双层，售价15元。",
    "芒果双皮奶在传统双皮奶上加芒果丁，售价18元。",
    "芋泥波波冰为本店2026年春季新品，芋头现蒸现捣，配黑糖波波，售价18元。",
    "红豆沙是冬季热饮首选，陈皮风味，售价12元。",
]

# ---- Step 1: 切割（按句切，每个 chunk 约一个完整事实）----
chunks = [s for d in docs for s in d.replace("。", "。|").split("|") if s.strip()]
print(f"Step1 切割：{len(docs)} 个文档 → {len(chunks)} 个 chunk（示例：{chunks[0][:18]}…）")

# ---- Step 2: 向量化（字符 bigram 词袋 + TF-IDF，模拟 Embedding）----
def bigrams(t):
    t = re.sub(r"[，。、！？\s]", "", t)
    return [t[i:i+2] for i in range(len(t) - 1)]

vocab = sorted({g for c in chunks for g in bigrams(c)})
gidx = {g: i for i, g in enumerate(vocab)}

def bow(t):
    v = np.zeros(len(vocab))
    for g in bigrams(t):
        if g in gidx:
            v[gidx[g]] += 1
    return v

X = np.stack([bow(c) for c in chunks])
df = (X > 0).sum(axis=0)
idf = np.log((1 + len(chunks)) / (1 + df)) + 1
Xw = np.log1p(X) * idf
Xn = Xw / np.linalg.norm(Xw, axis=1, keepdims=True)
print(f"Step2 向量化：每个 chunk → {len(vocab)} 维 TF-IDF 向量")

# ---- Step 3: 检索（余弦相似度 Top-K）----
def retrieve(query, k=3):
    q = bow(query) * idf
    qn = q / (np.linalg.norm(q) + 1e-9)
    sims = Xn @ qn
    order = np.argsort(-sims)
    return [(chunks[i], sims[i]) for i in order[:k]]

query = "芋泥波波冰多少钱"
print(f"\nStep3 检索：查询「{query}」")
for c, s in retrieve(query):
    if s > 0.05:
        print(f"   相似度 {s:.3f} | {c}")
print("   （其余 chunk 相似度为 0，无匹配）")

# ---- Step 4: 增强生成（把检索结果拼进上下文，再抽取答案）----
top_chunk = retrieve(query, 1)[0][0]
m = re.search(r"售价\d+元", top_chunk)
print(f"\nStep4 增强：把「{top_chunk[:20]}…」塞进 prompt，LLM 依据上下文作答")
print(f"   → 回答：芋泥波波冰 {m.group(0) if m else '（未找到）'} ✓ 与真实菜单一致")

## 实验 3：切割的暗坑 —— 事实被切断在 chunk 边界上

同样是"切割"，切得好检索就准，切得不好**关键信息正好跨在两个 chunk 的边界上**，
检索任何一半都拿不到完整事实。这就是 md 里"误区 2：文档丢进去就能用"的可执行版证据。

In [ ]:
import numpy as np

fact = "本店杨枝甘露使用泰国进口椰浆，售价22元，夏季限量供应。"
filler = "欢迎光临本店，扫码点单可享会员积分，营业时间为早十点到晚十点。"
doc = filler + fact + filler
query = "杨枝甘露多少钱"

def bigrams(t):
    return [t[i:i+2] for i in range(len(t) - 1)]

def score_chunks(chunks, query):
    vocab = sorted({g for c in chunks for g in bigrams(c)})
    gidx = {g: i for i, g in enumerate(vocab)}
    def vec(t):
        v = np.zeros(len(vocab))
        for g in bigrams(t):
            if g in gidx:
                v[gidx[g]] += 1
        return v
    q = vec(query)
    qn = q / (np.linalg.norm(q) + 1e-9)
    out = []
    for c in chunks:
        v = vec(c)
        vn = v / (np.linalg.norm(v) + 1e-9)
        out.append(float(qn @ vn))
    return out

# 切法A：无重叠，硬切 —— 关键句被拦腰斩断
cut = len(filler) + 14   # 故意切在"椰浆"和"售价"之间
chunks_a = [doc[:cut], doc[cut:cut+20], doc[cut+20:]]
# 切法B：带 50% 重叠 —— 边界处两边都保留完整上下文
chunks_b = [doc[:cut+12], doc[cut-12:cut+12], doc[cut:cut+24], doc[cut+12:]]

for name, chunks in [("A 无重叠", chunks_a), ("B 带12字重叠", chunks_b)]:
    scores = score_chunks(chunks, query)
    best = chunks[int(np.argmax(scores))]
    has_name, has_price = "杨枝甘露" in best, "22元" in best
    ok = has_name and has_price
    print(f"切法{name}：最优 chunk 长度 {len(best)} 字")
    print(f"   含「杨枝甘露」：{'✓' if has_name else '✗'}   含「22元」：{'✓' if has_price else '✗'}"
          f"   → {'✓ 可答' if ok else '✗ 事实被切断，答不了'}\n")

print("结论：chunk 边界会'吞掉'横跨两侧的事实；重叠切割（overlap）是最便宜的保险。")

## 实验 4：RAG 到底强在哪 —— 能力雷达（模拟示意）

用一组模拟数字直观对比：RAG 补的是**"你们店自己的、时效性的知识"**，
对纯推理/闲聊并没有增益（那是模型本身的能力）。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

# 模拟示意数据（非实测）：三类问题上的回答正确率
cats = ["时效知识\n(2026新菜单)", "店内事实\n(配料/价格)", "常识推理\n(无需资料)"]
closed_book = [0.15, 0.40, 0.85]
with_rag    = [0.95, 0.90, 0.85]

x = np.arange(len(cats))
w = 0.35
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.bar(x - w/2, closed_book, w, label="闭卷 LLM", color="#c0504d")
ax.bar(x + w/2, with_rag, w, label="RAG（检索增强）", color="#4f81bd")
for i in range(len(cats)):
    ax.text(x[i] - w/2, closed_book[i] + 0.02, f"{closed_book[i]:.0%}", ha="center", fontsize=9)
    ax.text(x[i] + w/2, with_rag[i] + 0.02, f"{with_rag[i]:.0%}", ha="center", fontsize=9)
ax.set_xticks(x, cats)
ax.set_ylabel("回答正确率（模拟示意）")
ax.set_ylim(0, 1.1)
ax.set_title("RAG 补的是知识，不是推理：两类知识型问题大幅提升，推理类无变化")
ax.legend()
plt.tight_layout()
plt.show()

print("读图：RAG 的增益集中在左两列（外部/时效知识）；最右列提醒我们——")
print("     别指望 RAG 替代模型能力，也别指望微调替代知识库（各管各的）。")

## 结论

| 问题 | 实验证据 |
|---|---|
| 为什么需要 RAG | 实验1：闭卷模型对没见过的知识只能编 |
| 四步管道每步做什么 | 实验2：切割→TF-IDF向量化→余弦Top-K→拼上下文，15行代码跑通 |
| 切割为什么重要 | 实验3：事实跨 chunk 边界 → 检索拿到半截信息；overlap 是保险 |
| RAG 的边界 | 实验4：提升知识型问题，不提升推理型问题 |

→ 深入阅读：`ima/第4周-Day1-RAG基本流程与架构.md`（组件技术栈、糖水店业务场景、4 个常见误区）